# 01 · Total aerial greenery per ward
Green area (London Green Cover map, from aerial images) inside each 2018 ward polygon, as a share of its area: the NDVI measure. LSOA twin: `01_SAT_greenery_total_lsoa.ipynb`.
`ward_area` is **land** area: ward polygons extend into the tidal Thames, so the shapefile's `NONLD_AREA` (water, ha) is taken out of the denominator, as the LSOA boundaries already are.

**Inputs:** `data/raw_data/green_cover_tiles/ndvi_landcover_merge_*.shp` (not included, about 13 GB; see the README), ward boundaries.
**Output:** `data/greenery_metrics/SAT_greenery_total_ward.csv`.
**Cache:** `data/greenery_metrics/SAT_greenery_total_ward_chunks/{tile_id}.csv`, one per tile; a rerun resumes after a crash. Set `FORCE_REBUILD = True` to recompute.
**Logic:** `WARD_CONFIG` in `01_SAT_greenery_total_workers.py`.

In [ ]:
import importlib
import os
import sys
from pathlib import Path

from joblib import Parallel, delayed

sys.path.insert(0, str(Path.cwd().resolve()))
gtw = importlib.import_module("01_SAT_greenery_total_workers")   # file name starts with a digit
CONFIG = gtw.WARD_CONFIG
N_JOBS = min(gtw.N_BUCKETS, os.cpu_count() or 1)
FORCE_REBUILD = False   # True: recompute every tile
gtw.print_status(CONFIG)
bucket_ids = gtw.buckets_to_run(CONFIG, force=FORCE_REBUILD)
print(f"buckets to run: {len(bucket_ids)}")

## 1. Tiles in parallel

In [ ]:
if bucket_ids:
    Parallel(n_jobs=N_JOBS, backend="loky", verbose=10)(
        delayed(gtw.run_bucket)(thread_id, CONFIG, force=FORCE_REBUILD) for thread_id in bucket_ids)
gtw.print_status(CONFIG)

## 2. Combine the tiles → SAVE

In [ ]:
out = gtw.finalize_total(CONFIG)
print(f"wrote {len(out)} rows → {gtw.write_total_csv(out, CONFIG)}")
out.head()

## Validation

In [ ]:
assert list(out.columns) == [CONFIG.area_id_col, CONFIG.area_area_col, "total_green_area", "percent_green"]
assert len(out) == 657 and out[CONFIG.area_id_col].is_unique and out.notna().all().all()
assert (out["total_green_area"] > 0).all() and out["percent_green"].between(0, 1).all()
print(f"percent_green: {out['percent_green'].min():.4f} – {out['percent_green'].max():.4f}")